# Tutorial: The Data Science Landscape

> **Type:** Tutorial. Follow along step by step. Your only job is to make it work.  
> **Time:** ~1.5 hours  
> **Prerequisites:** Python basics, pip  
> **What you'll build:** An interactive timeline of AI milestones, plus your first data analysis: 30 years of machine learning paper publication trends on arXiv.

---

## What We're Building

We're going to do two things:

1. Build a visual timeline of the AI milestones from `explanation.md`. A picture of history is easier to hold in your head than a list.
2. Analyse a dataset of arXiv machine learning paper counts. You'll see, with numbers, when different subfields took off, and you'll use basic pandas operations to find the story in the data.

This is also your first taste of the core data science workflow: **load → look → clean → visualise → interpret**. We'll do all five steps here, and we'll repeat them in every module that follows.

## Step 0: Setup

In [ ]:
# Install dependencies (uncomment if running for the first time)
# !pip install pandas matplotlib seaborn requests

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import json
from datetime import datetime

# Make plots look good
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print('Setup complete.')

## Part 1: The AI Timeline

Let's build the timeline from `explanation.md` as a proper visualisation. We'll define the milestones ourselves. That gives you practice creating structured data in Python before you load any external datasets.

### Step 1: Define the Milestones

In [ ]:
# Each milestone has a year, a label, and an era
# Era determines the colour on the chart
milestones = [
    # Era 1: Rules & Expert Systems
    {"year": 1950, "label": "Turing Test proposed", "era": "Rules & Expert Systems"},
    {"year": 1956, "label": "'Artificial Intelligence' coined\n(Dartmouth Conference)", "era": "Rules & Expert Systems"},
    {"year": 1965, "label": "DENDRAL: first expert system", "era": "Rules & Expert Systems"},
    {"year": 1972, "label": "MYCIN: medical diagnosis AI", "era": "Rules & Expert Systems"},
    
    # Era 2: Statistical ML
    {"year": 1986, "label": "Backpropagation\nrediscovered", "era": "Statistical ML"},
    {"year": 1995, "label": "Support Vector Machines", "era": "Statistical ML"},
    {"year": 1997, "label": "LSTM invented", "era": "Statistical ML"},
    {"year": 2001, "label": "Random Forests", "era": "Statistical ML"},
    {"year": 2001, "label": "AdaBoost → Gradient Boosting", "era": "Statistical ML"},
    {"year": 2009, "label": "ImageNet dataset released", "era": "Statistical ML"},
    
    # Era 3: Deep Learning & Foundation Models
    {"year": 2012, "label": "AlexNet wins ImageNet", "era": "Deep Learning & Foundation Models"},
    {"year": 2014, "label": "GANs", "era": "Deep Learning & Foundation Models"},
    {"year": 2017, "label": "Transformer architecture\n('Attention Is All You Need')", "era": "Deep Learning & Foundation Models"},
    {"year": 2018, "label": "BERT (Google)", "era": "Deep Learning & Foundation Models"},
    {"year": 2020, "label": "GPT-3", "era": "Deep Learning & Foundation Models"},
    {"year": 2021, "label": "GitHub Copilot", "era": "Deep Learning & Foundation Models"},
    {"year": 2022, "label": "ChatGPT", "era": "Deep Learning & Foundation Models"},
    {"year": 2023, "label": "GPT-4, Llama, Claude", "era": "Deep Learning & Foundation Models"},
]

df_milestones = pd.DataFrame(milestones)
print(f"{len(df_milestones)} milestones across {df_milestones['era'].nunique()} eras")
df_milestones.head()

### Step 2: Plot the Timeline

In [ ]:
era_colours = {
    "Rules & Expert Systems":         "#4E79A7",
    "Statistical ML":                  "#F28E2B",
    "Deep Learning & Foundation Models": "#E15759",
}

fig, ax = plt.subplots(figsize=(16, 5))

# Draw the baseline
ax.axhline(0, color='black', linewidth=1.5, zorder=1)

# Alternate label positions above/below the line to avoid overlaps
positions = [0.6, 1.1, 1.6, 2.1, -0.6, -1.1, -1.6]

for i, row in df_milestones.iterrows():
    colour = era_colours[row['era']]
    y_pos = positions[i % len(positions)]
    
    # Dot on the line
    ax.scatter(row['year'], 0, color=colour, s=80, zorder=3)
    
    # Vertical connector
    ax.plot([row['year'], row['year']], [0, y_pos * 0.85], 
            color=colour, linewidth=1, linestyle='--', alpha=0.6, zorder=2)
    
    # Label
    ax.text(row['year'], y_pos, row['label'],
            ha='center', va='center', fontsize=8,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', 
                      edgecolor=colour, linewidth=1.2))

# Era background shading
ax.axvspan(1950, 1984, alpha=0.04, color=era_colours["Rules & Expert Systems"])
ax.axvspan(1984, 2011, alpha=0.04, color=era_colours["Statistical ML"])
ax.axvspan(2011, 2025, alpha=0.04, color=era_colours["Deep Learning & Foundation Models"])

# Legend
legend_patches = [mpatches.Patch(color=c, label=e) for e, c in era_colours.items()]
ax.legend(handles=legend_patches, loc='lower right', fontsize=9)

ax.set_xlim(1945, 2026)
ax.set_ylim(-2.5, 2.5)
ax.set_yticks([])
ax.set_xlabel('Year', fontsize=12)
ax.set_title('Key Milestones in AI / Data Science', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('ai_timeline.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to ai_timeline.png')

> **What just happened?** We built a structured dataset from scratch, assigned categories (eras), and visualised it. You'll use the same pattern for building features and plotting distributions throughout the course.

---

## Part 2: Exploring arXiv ML Paper Trends

Now we'll look at paper counts. We'll use real arXiv counts by year and subfield. They show when different parts of the field took off.

arXiv has no "deep learning" or "classical ML" category, so each count is the number of papers submitted that year whose abstract mentions at least one of the subfield's key phrases. The phrases are listed in the next cell. A paper can count in more than one subfield. The counts were pulled from the arXiv API on 2026-11-17. The optional cell after the data shows how to pull them again.

### Step 3: Load the Data

In [ ]:
# Real arXiv paper counts per year per ML subfield, 1993-2024.
# Count = papers submitted that year whose abstract mentions any of:
#   classical_ml             support vector machine, random forest, logistic regression, decision tree, gradient boosting, naive bayes, nearest neighbor
#   deep_learning            deep learning, deep neural network, convolutional neural network, recurrent neural network
#   nlp_llm                  natural language processing, language model
#   computer_vision          computer vision, image classification, object detection, image segmentation
#   reinforcement_learning   reinforcement learning
# Source: arXiv API (export.arxiv.org/api/query), pulled 2026-11-17.
data = {
    'year': list(range(1993, 2025)),
    'classical_ml': [
        29, 52, 65, 76, 107, 122, 123, 146,
        136, 154, 172, 237, 243, 269, 321, 378,
        341, 457, 510, 577, 612, 730, 794, 1014,
        1344, 1607, 1967, 2258, 2291, 2273, 2405, 2850,
    ],
    'deep_learning': [
        0, 1, 0, 0, 1, 3, 2, 5,
        1, 0, 2, 5, 3, 3, 8, 7,
        4, 4, 10, 27, 107, 275, 933, 2024,
        4103, 7176, 9575, 12219, 12500, 12133, 12813, 13624,
    ],
    'nlp_llm': [
        0, 21, 24, 26, 26, 13, 6, 27,
        14, 8, 6, 5, 4, 5, 8, 12,
        16, 13, 33, 52, 78, 134, 199, 362,
        520, 762, 1334, 2206, 3065, 4253, 11447, 24238,
    ],
    'computer_vision': [
        1, 0, 7, 6, 2, 10, 19, 9,
        13, 10, 15, 20, 23, 21, 34, 41,
        57, 67, 76, 148, 216, 349, 584, 848,
        1325, 2029, 2758, 3609, 4353, 4780, 5643, 6095,
    ],
    'reinforcement_learning': [
        0, 0, 2, 1, 0, 1, 7, 1,
        4, 6, 6, 4, 4, 9, 4, 14,
        14, 21, 46, 70, 67, 75, 115, 313,
        739, 1341, 2231, 3010, 3325, 3680, 4444, 5533,
    ],
}

df = pd.DataFrame(data)
print(f"Dataset shape: {df.shape}")
print(f"Years covered: {df['year'].min()} – {df['year'].max()}")
df.head(10)

In [ ]:
# Optional: pull the counts again from the arXiv API.
# Takes about 10 minutes for 5 subfields x 32 years: arXiv asks for
# no more than one request every 3 seconds. Off by default.
import re, time, urllib.parse, urllib.request

FETCH = False
PHRASES = {
    'classical_ml': ['support vector machine', 'random forest', 'logistic regression', 'decision tree', 'gradient boosting', 'naive bayes', 'nearest neighbor'],
    'deep_learning': ['deep learning', 'deep neural network', 'convolutional neural network', 'recurrent neural network'],
    'nlp_llm': ['natural language processing', 'language model'],
    'computer_vision': ['computer vision', 'image classification', 'object detection', 'image segmentation'],
    'reinforcement_learning': ['reinforcement learning'],
}

def arxiv_count(phrases, year):
    dates = f'submittedDate:[{year}01010000 TO {year}12312359]'
    terms = ' OR '.join(f'abs:"{p}"' for p in phrases)
    query = urllib.parse.urlencode(
        {'search_query': f'({terms}) AND {dates}', 'max_results': 1}
    )
    url = 'https://export.arxiv.org/api/query?' + query
    req = urllib.request.Request(url, headers={'User-Agent': 'notebook'})
    feed = urllib.request.urlopen(req, timeout=60).read().decode()
    return int(re.search(r'<opensearch:totalResults>(\d+)', feed).group(1))

if FETCH:
    for col, phrases in PHRASES.items():
        for year in df['year']:
            df.loc[df['year'] == year, col] = arxiv_count(phrases, year)
            time.sleep(3)

### Step 4: Understand the Data

Before plotting anything, always look at the shape, the types, and basic statistics. This habit will save you hours of debugging.

In [ ]:
# Basic inspection
print("=== Data types ===")
print(df.dtypes)

print("\n=== Missing values ===")
print(df.isnull().sum())

print("\n=== Summary statistics ===")
df.describe().round(1)

In [ ]:
# Which subfield had the most papers in the most recent year?
latest_year = df[df['year'] == df['year'].max()]
subfields = ['classical_ml', 'deep_learning', 'nlp_llm', 'computer_vision', 'reinforcement_learning']

print(f"Paper counts in {df['year'].max()}:")
for col in subfields:
    print(f"  {col:35s}: {latest_year[col].values[0]:>8,}")

### Step 5: Visualise the Trends

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colours = ['#4E79A7', '#E15759', '#F28E2B', '#76B7B2', '#59A14F']
labels  = ['Classical ML', 'Deep Learning', 'NLP / LLMs', 'Computer Vision', 'Reinforcement Learning']

# --- Left: Full history ---
for col, colour, label in zip(subfields, colours, labels):
    axes[0].plot(df['year'], df[col], label=label, color=colour, linewidth=2)

# Mark the transformer paper
axes[0].axvline(2017, color='grey', linestyle=':', linewidth=1.5)
axes[0].text(2017.2, axes[0].get_ylim()[1] * 0.85, 'Transformer\n(2017)', 
             fontsize=9, color='grey')

axes[0].set_title('ML Paper Volume by Subfield\n(all years)', fontweight='bold')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Papers per year (abstract mentions)')
axes[0].legend(fontsize=9)

# --- Right: Zoom into 2012–2024 ---
df_modern = df[df['year'] >= 2012].copy()

for col, colour, label in zip(subfields, colours, labels):
    axes[1].plot(df_modern['year'], df_modern[col], label=label, color=colour, 
                 linewidth=2, marker='o', markersize=4)

axes[1].axvline(2017, color='grey', linestyle=':', linewidth=1.5)
axes[1].axvline(2022, color='darkred', linestyle=':', linewidth=1.5)
axes[1].text(2017.1, axes[1].get_ylim()[1] * 0.95, 'Transformer', fontsize=8, color='grey')
axes[1].text(2022.1, axes[1].get_ylim()[1] * 0.85, 'ChatGPT', fontsize=8, color='darkred')

axes[1].set_title('ML Paper Volume by Subfield\n(2012–2024 zoom)', fontweight='bold')
axes[1].set_xlabel('Year')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('arxiv_trends.png', dpi=150, bbox_inches='tight')
plt.show()

> **What just happened?** You just did your first data analysis: load → inspect → visualise → interpret. The chart shows three things. Deep learning takes off after 2012 and flattens out after 2020. Language-model papers explode after ChatGPT's release in late 2022, more than doubling in both 2023 and 2024. Classical ML keeps growing steadily. It isn't being replaced. It's being outgrown.

### Step 6: Answer a Specific Question From the Data

Data analysis isn't only plotting. It's answering questions. Let's answer one: **in what year did deep learning paper volume first exceed classical ML?**

In [ ]:
# Find the crossover year
df['dl_vs_classical'] = df['deep_learning'] - df['classical_ml']
df = df.sort_values('year')  # .iloc[0] needs year order
crossover = df[df['dl_vs_classical'] > 0].iloc[0]

print(f"Deep learning overtook classical ML in: {int(crossover['year'])}")
print(f"  Classical ML papers that year: {crossover['classical_ml']:,}")
print(f"  Deep learning papers that year: {crossover['deep_learning']:,}")

# What was the ratio by 2024?
row_2024 = df[df['year'] == 2024].iloc[0]
ratio = row_2024['deep_learning'] / row_2024['classical_ml']
print(f"\nBy 2024, deep learning papers outnumber classical ML by {ratio:.1f}x")

### Step 7: Year-on-Year Growth Rate

In [ ]:
# Calculate year-on-year percentage growth for each subfield
growth = df[['year'] + subfields].copy()
for col in subfields:
    growth[f'{col}_yoy'] = growth[col].pct_change() * 100

# Show peak growth years for NLP/LLMs
nlp_growth = growth[['year', 'nlp_llm_yoy']].dropna()
peak_growth_years = nlp_growth.nlargest(5, 'nlp_llm_yoy')

print("Top 5 years for NLP/LLM paper growth:")
print(peak_growth_years.to_string(index=False))

In [ ]:
# Exclude the inf artifact before ranking growth years
finite_growth = nlp_growth[nlp_growth['nlp_llm_yoy'] != float('inf')]
peak_finite = finite_growth.nlargest(5, 'nlp_llm_yoy')
print("Top 5 years for NLP/LLM paper growth, inf excluded:")
print(peak_finite.to_string(index=False))

> **What just happened?** With the `inf` artefact excluded, 2000 is the largest growth year (350%), but that's a low-base effect: 6 papers to 27. The first big jump on a large base is 2023 (169%, from 4,253 to 11,447 papers), the year after ChatGPT's release. 2024 follows at 112%.

---

## What You Built

You built two things: a visual history of AI from structured data you defined yourself, and a trend analysis of ML research growth. More importantly, you ran through the core data science loop, **load → inspect → clean → visualise → interpret**, for the first time. Every later module repeats this loop on a different domain.

## What's Next

- Read `explanation.md` if you haven't. The historical context makes the trend charts much more meaningful.
- Open `howto.md` for pandas and matplotlib recipes you'll reuse constantly.
- Keep `reference.md` open as a pandas cheat sheet.
- **Extension challenge:** add a 6th subfield to the milestone list: generative AI and diffusion models. Add it to the DataFrame and update both charts.

---

**Next module:** `02_business_problem_framing`, where you learn to turn a customer's request into a data science problem statement before writing a single line of code.